# Soup Kernel Challenge: developer check (Kaggle T4)

Fast checks of the solution before the full measured run (`templates/kaggle_notebook.ipynb`):
1. kernel unit tests against PyTorch / bitsandbytes (`pytest solution`),
2. end-to-end comparison of the fast path with the stock transformers+PEFT step on the real model (`dev/check_fast_path.py`: loss, entropy, accuracy, rewind losses, all LoRA gradients, timing, memory),
3. one measured `soup train` run of the solution alone (`harness/run_task.py --only solution`).

Settings: **GPU T4 x2**, **Internet on**. Put the repository URL in the first cell, then Run All (~8 min).

In [ ]:
REPO_URL = "https://github.com/<you>/<your-repo>.git"  # your repository
COMMIT = ""  # empty = default branch

In [ ]:
import os, subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
!python -m pip install -q "soup-cli[train]==0.77.0" "transformers==5.16.1" "peft==0.20.0" "trl==0.29.1" "bitsandbytes==0.50.2" "accelerate==1.14.0" pytest

In [ ]:
import base64, shutil, subprocess
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as exc:
    print("No GITHUB_TOKEN secret:", type(exc).__name__)
auth = []
if token:
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    auth = ["-c", f"http.extraHeader=Authorization: Basic {basic}"]
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)
r = subprocess.run(["git", *auth, "clone", "-q", REPO_URL, "/kaggle/working/repo"], capture_output=True, text=True)
if r.returncode: raise SystemExit(r.stderr)
if COMMIT: subprocess.run(["git", "-C", "/kaggle/working/repo", "checkout", "-q", COMMIT], check=True)
print(subprocess.run(["git", "-C", "/kaggle/working/repo", "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)

In [ ]:
!cd /kaggle/working/repo && python task/prepare_data.py
!cd /kaggle/working/repo && python -m pytest solution -q -x 2>&1 | tail -30

In [ ]:
!cd /kaggle/working/repo && python dev/check_fast_path.py 2>&1 | grep -v Warning | tail -150

In [ ]:
!cd /kaggle/working/repo && python harness/run_task.py --only solution --out runs_try 2>&1 | tail -8
!cd /kaggle/working/repo && grep "\[solution\]" runs_try/solution_1/train.log | head -30
!cd /kaggle/working/repo && tail -25 runs_try/solution_1/train.log

In [ ]:
import json
r = json.load(open("/kaggle/working/repo/runs_try/result.json"))
run = r["runs"]["solution"][0]
m = run["metrics"]
own = sum(v for k, v in m["own_kernel_ms"].items())
print({"wall_s": run["wall_s"], "gpu_peak_mib": run["gpu_peak_mib"], "train_loop_s": m["train_loop_s"],
       "median_step_s": m["median_step_s"], "apply_s": m["apply_s"], "torch_max_alloc_mib": m["torch_max_allocated_mib"],
       "torch_max_reserved_mib": m["torch_max_reserved_mib"], "own_kernel_share": round(own / 1000 / m["timed_loop_s"], 3),
       "eval_loss": r["evals"].get("solution", {}).get("mean_loss")})
print("own kernel ms:", m["own_kernel_ms"])
print("loss curve:", run["log"]["loss"][:8], "...")